## Merge

In [ ]:
import dask.dataframe as dd
import os

linked_features = ['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'order_time_jittered_utc']

files_info_processed = [
    {
        "name": "processed_cultures_cohort",
        "path": "microbiology_cultures_cohort.csv",
        "merge_on": linked_features, 
        "dtype": {
            "anon_id":"object",
            "pat_enc_csn_id_coded":"int64",
            "order_proc_id_coded":"int64 ",
            "order_time_jittered_utc":"object",
            "organism": "int8",
            "antibiotic": "int8",
            "susceptibility": "int8"
        }
    },
    {
        "name": "processed_prior_med",
        "path": "microbiology_cultures_prior_med.csv",
        "merge_on": linked_features,
        "dtype": {
             "anon_id": "object",
            "pat_enc_csn_id_coded": "int64",
            "order_proc_id_coded": "int64",
            "order_time_jittered_utc": "object",
            "medication_time_to_culturetime": "Int64", 
            "medication_category": "int8",
            "med_within_30_days": "boolean"
        }
    },
    {
        "name": "processed_cultures_demographics",
        "path": "microbiology_cultures_demographics.csv",
        "merge_on": linked_features[:3], 
        "dtype": {
            "anon_id":"object",
            "pat_enc_csn_id_coded":"int64",
            "order_proc_id_coded":"int64 ",
            "age": "int8",
            "gender": "int8"
        }
    },
     {
        "name": "processed_cultures_labs",
        "path": "microbiology_cultures_labs.csv",
        "merge_on": linked_features[:3], 
        "dtype": {
            "anon_id": "object",
            "pat_enc_csn_id_coded": "int64",
            "order_proc_id_coded": "int64",
            "median_wbc": "float32",
            "median_neutrophils": "float32",
            "median_lymphocytes": "float32",
            "median_hgb": "float32",
            "median_plt": "float32",
            "median_na": "float32",
            "median_hco3": "float32",
            "median_bun": "float32",
            "median_cr": "float32",
            "median_lactate": "float32",
            "median_procalcitonin": "float32",
            "delta_wbc": "float32",
            "delta_na": "float32",
            "delta_hco3": "float32",
            "delta_bun": "float32",
            "delta_cr": "float32"
        }
    },
    {
        "name": "processed_cultures_vitals",
        "path": "microbiology_cultures_vitals.csv",
        "merge_on": linked_features[:3],
        "dtype": {
            "anon_id": "object",
            "pat_enc_csn_id_coded": "int64",
            "order_proc_id_coded": "int64",
            "median_heartrate": "float32",
            "median_resprate": "float32",
            "median_temp": "float32",
            "median_sysbp": "float32",
            "median_diasbp": "float32",
            "delta_heartrate": "float32",
            "delta_resprate": "float32",
            "delta_temp": "float32",
            "delta_sysbp": "float32",
            "delta_diasbp": "float32"
        }
    },
     {
        "name": "processed_antibiotic_class_exposure",
        "path": "microbiology_cultures_antibiotic_class_exposure.csv",
        "merge_on": linked_features, 
        "dtype": {
            "anon_id": "object",
            "pat_enc_csn_id_coded": "int64",
            "order_proc_id_coded": "int64",
            "order_time_jittered_utc": "object",
            "antibiotic_class": "int8",
            "time_to_culturetime": "Int64"  
        }
    },
    {
    "name": "processed_cultures_comorbidity",
    "path": "microbiology_cultures_comorbidity.csv",
    "merge_on": linked_features, 
    "dtype": {
        "order_proc_id_coded": "int64",
        "has_diabetes": "int8",
        "has_chf": "int8",
        "has_ckd": "int8",
        "has_cancer": "int8",
        "has_transplant": "int8",
        "has_immunosuppression": "int8",
        "has_copd": "int8",
        "has_liver_disease": "int8",
        "has_dementia": "int8",
        "has_stroke": "int8",
        "has_other_comorbid": "int8"
    }
}

]

sample_folder = 'new_sample_one_processed/'
merged_folder = 'merged_sample_one_processed/'
os.makedirs(merged_folder, exist_ok=True)         

file_df_pairs = [file['name'] + '.parquet' for file in files_info_processed]

based_df_dtype = files_info_processed[0]['dtype']
df_dtype = files_info_processed[1]['dtype']

based_df = dd.read_parquet(sample_folder + file_df_pairs[0], 
                           dtype = based_df_dtype)
df = dd.read_parquet(sample_folder + file_df_pairs[1], 
                           dtype = df_dtype )

based_df = based_df.merge(df, how="left", on=files_info_processed[0]['merge_on'])

print(f"Start merging: left {files_info_processed[0]['name']}.parquet ; right {files_info_processed[1]['name']}.parquet")

based_df.to_parquet(
            merged_folder + "stage01/",
            engine='pyarrow',
            compression='snappy',
            write_index=False
        )

based_df_dtype.update(df_dtype)

print(f"Merged result saved - Cols # {len(based_df.columns)}")

Start merging: left processed_cultures_cohort.parquet ; right processed_prior_med.parquet
Merged result saved - Cols # 10


In [ ]:
based_df.columns

Index(['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded',
       'order_time_jittered_utc', 'organism', 'antibiotic', 'susceptibility',
       'med_within_30_days', 'n_med_categories', 'min_time_to_med'],
      dtype='str')

In [ ]:

df_dtype = files_info_processed[2]['dtype']
based_df = dd.read_parquet(merged_folder + "stage01/" +"*.parquet", 
                           dtype = based_df_dtype)
df = dd.read_parquet(sample_folder + file_df_pairs[2], 
                           dtype = df_dtype )
# print(based_df.info())  # Check the structure before merging

based_df = based_df.merge(df, how='left', on=files_info_processed[2]['merge_on'])

print(f"Start merging: left Based File ; right {files_info_processed[2]['name']}.parquet")

based_df.to_parquet(
    merged_folder + "stage02/",
    engine="pyarrow",
    compression="snappy",
    write_index=False,
    overwrite=True
)

based_df_dtype.update(df_dtype)

print(f"Merged result saved - Cols # {len(based_df.columns)}")

Start merging: left Based File ; right processed_cultures_demographics.parquet
Merged result saved - Cols # 12


In [ ]:

df_dtype = files_info_processed[3]['dtype']
based_df = dd.read_parquet(merged_folder + "stage02/" +"*.parquet", 
                           dtype = based_df_dtype)
df = dd.read_parquet(sample_folder + file_df_pairs[3], 
                           dtype = df_dtype )

based_df = based_df.merge(df, how='left', on=files_info_processed[3]['merge_on'])

print(f"Start merging: left Based File ; right {files_info_processed[3]['name']}.parquet")

based_df.to_parquet(
            merged_folder + "stage03/",
            engine='pyarrow',
            compression='snappy',
            write_index=False,
            overwrite=True
        )

based_df_dtype.update(df_dtype)

print(f"Merged result saved - Cols # {len(based_df.columns)}")

Start merging: left Based File ; right processed_cultures_labs.parquet
Merged result saved - Cols # 28


In [ ]:

df_dtype = files_info_processed[4]['dtype']
based_df = dd.read_parquet(merged_folder + "stage03/" +"*.parquet", 
                           dtype = based_df_dtype)
df = dd.read_parquet(sample_folder + file_df_pairs[4], 
                           dtype = df_dtype )

based_df = based_df.merge(df, how='left', on=files_info_processed[4]['merge_on'])

print(f"Start merging: left Based File ; right {files_info_processed[4]['name']}.parquet")

based_df.to_parquet(
            merged_folder + "stage04/",
            engine='pyarrow',
            compression='snappy',
            write_index=False,
            overwrite=True
        )

based_df_dtype.update(df_dtype)

print(f"Merged result saved - Cols # {len(based_df.columns)}")

Start merging: left Based File ; right processed_cultures_vitals.parquet
Merged result saved - Cols # 48


## check the merge result until now!

In [ ]:
based_df = dd.read_parquet(merged_folder + "stage04/" +"*.parquet", 
                           dtype = based_df_dtype)
num_cols = len(based_df.columns)
num_rows = based_df.shape[0].compute()
print(f"Shape: ({num_rows},{num_cols})")

Shape: (455826,48)


In [ ]:
import dask.dataframe as dd
import os

linked_features = ['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'order_time_jittered_utc']

files_info_processed = [
    {
        "name": "processed_cultures_cohort",
        "path": "microbiology_cultures_cohort.csv",
        "merge_on": linked_features, 
        "dtype": {
            "anon_id":"object",
            "pat_enc_csn_id_coded":"int64",
            "order_proc_id_coded":"int64 ",
            "order_time_jittered_utc":"object",
            "organism": "int8",
            "antibiotic": "int8",
            "susceptibility": "int8"
        }
    },
    {
        "name": "processed_prior_med",
        "path": "microbiology_cultures_prior_med.csv",
        "merge_on": linked_features,
        "dtype": {
             "anon_id": "object",
            "pat_enc_csn_id_coded": "int64",
            "order_proc_id_coded": "int64",
            "order_time_jittered_utc": "object",
            "medication_time_to_culturetime": "Int64", 
            "medication_category": "int8",
            "med_within_30_days": "boolean"
        }
    },
    {
        "name": "processed_cultures_demographics",
        "path": "microbiology_cultures_demographics.csv",
        "merge_on": linked_features[:3], 
        "dtype": {
            "anon_id":"object",
            "pat_enc_csn_id_coded":"int64",
            "order_proc_id_coded":"int64 ",
            "age": "int8",
            "gender": "int8"
        }
    },
     {
        "name": "processed_cultures_labs",
        "path": "microbiology_cultures_labs.csv",
        "merge_on": linked_features[:3], 
        "dtype": {
            "anon_id": "object",
            "pat_enc_csn_id_coded": "int64",
            "order_proc_id_coded": "int64",
            "median_wbc": "float32",
            "median_neutrophils": "float32",
            "median_lymphocytes": "float32",
            "median_hgb": "float32",
            "median_plt": "float32",
            "median_na": "float32",
            "median_hco3": "float32",
            "median_bun": "float32",
            "median_cr": "float32",
            "median_lactate": "float32",
            "median_procalcitonin": "float32",
            "delta_wbc": "float32",
            "delta_na": "float32",
            "delta_hco3": "float32",
            "delta_bun": "float32",
            "delta_cr": "float32"
        }
    },
    {
        "name": "processed_cultures_vitals",
        "path": "microbiology_cultures_vitals.csv",
        "merge_on": linked_features[:3],
        "dtype": {
            "anon_id": "object",
            "pat_enc_csn_id_coded": "int64",
            "order_proc_id_coded": "int64",
            "median_heartrate": "float32",
            "median_resprate": "float32",
            "median_temp": "float32",
            "median_sysbp": "float32",
            "median_diasbp": "float32",
            "delta_heartrate": "float32",
            "delta_resprate": "float32",
            "delta_temp": "float32",
            "delta_sysbp": "float32",
            "delta_diasbp": "float32"
        }
    },
     {
        "name": "processed_antibiotic_class_exposure",
        "path": "microbiology_cultures_antibiotic_class_exposure.csv",
        "merge_on": linked_features, 
        "dtype": {
            "anon_id": "object",
            "pat_enc_csn_id_coded": "int64",
            "order_proc_id_coded": "int64",
            "order_time_jittered_utc": "object",
            "antibiotic_class": "int8",
            "time_to_culturetime": "Int64"  
        }
    },
    {
    "name": "processed_cultures_comorbidity",
    "path": "microbiology_cultures_comorbidity.csv",
    "merge_on": linked_features, 
    "dtype": {
        "order_proc_id_coded": "int64",
        "has_diabetes": "int8",
        "has_chf": "int8",
        "has_ckd": "int8",
        "has_cancer": "int8",
        "has_transplant": "int8",
        "has_immunosuppression": "int8",
        "has_copd": "int8",
        "has_liver_disease": "int8",
        "has_dementia": "int8",
        "has_stroke": "int8",
        "has_other_comorbid": "int8"
    }
}

]

In [ ]:
import dask.dataframe as dd
import os

sample_folder = 'new_sample_one_processed/'
merged_folder = 'merged_sample_one_processed/'
os.makedirs(merged_folder, exist_ok=True)  

based_df_dtype = {}
for f in files_info_processed[:5]:
    based_df_dtype.update(f['dtype'])

df_dtype = files_info_processed[5]['dtype']
based_df = dd.read_parquet(merged_folder + "stage04/" +"*.parquet", 
                           dtype = based_df_dtype)
df = dd.read_parquet(sample_folder + files_info_processed[5]['name'] + '.parquet', 
                           dtype = df_dtype )

based_df = based_df.merge(df, how='left', on=files_info_processed[5]['merge_on'])

print(f"Start merging: left Based File ; right {files_info_processed[5]['name']}.parquet")

based_df.to_parquet(
            merged_folder + "stage05/",
            engine='pyarrow',
            compression='snappy',
            write_index=False,
            overwrite=True
        )

based_df_dtype.update(df_dtype)

print(f"Merged result saved - Cols # {len(based_df.columns)}")

Start merging: left Based File ; right processed_antibiotic_class_exposure.parquet
Merged result saved - Cols # 50


In [ ]:

df_dtype = files_info_processed[6]['dtype']
based_df = dd.read_parquet(merged_folder + "stage05/" +"*.parquet", 
                           dtype = based_df_dtype)
df = dd.read_parquet(sample_folder + files_info_processed[6]['name'] + '.parquet', 
                           dtype = df_dtype )

based_df = based_df.merge(df, how='left', on=files_info_processed[6]['merge_on'])

print(f"Start merging: left Based File ; right {files_info_processed[6]['name']}.parquet")

based_df.to_parquet(
            merged_folder + "final_stage/",
            engine='pyarrow',
            compression='snappy',
            write_index=False,
            overwrite=True
        )

based_df_dtype.update(df_dtype)

print(f"Merged result saved - Cols # {len(based_df.columns)}")

Start merging: left Based File ; right processed_cultures_comorbidity.parquet
Merged result saved - Cols # 61
